# 07 · Mixed sentiment

The model was trained for two classes, but many reviews are both: good location, dirty room. This notebook scores each **sentence** separately with the same fine-tuned AraBERT model, and uses the sentence scores to sort every test review into four classes: **positive, negative, mixed, neutral**.

| | |
|---|---|
| **Input** | `data/gold/test_predictions` (from `06`), model from the Hugging Face Hub |
| **Output** | `data/gold/sentence_predictions`, `data/gold/review_sentiment_4class` |

> There are no human labels for "mixed" or "neutral" in HARD, so these classes can't be scored for accuracy. They are a rule-based layer on top of the model, checked for sense against the star ratings below.

In [1]:
import time
from config import get_spark, GOLD_PATH
from pyspark.sql import functions as F
from huggingface_hub import snapshot_download

MODEL_DIR = snapshot_download("Mahmoud-103/arabert-hard-sentiment")
spark = get_spark("mixed")

fact = spark.read.parquet(f"{GOLD_PATH}/test_predictions")

## Split reviews into sentences

Split on sentence punctuation (`. ! ? ؟ ؛` and line breaks) and also on the Arabic words for "but" (`لكن`, `ولكن`, and the colloquial `بس`), since that is where reviewers usually switch from praise to complaint. `posexplode` keeps each sentence's position in the review. Fragments of one word are dropped.

The 10,217 test reviews give 25,466 sentences.

In [2]:
SAMPLE = 1.0

SPLIT = r"[.!?؟؛\n]+|\s+(?:لكن|ولكن|بس)\s+"

sents = (fact.sample(fraction=SAMPLE, seed=42)
    .select("review_id", F.posexplode(F.split("review_clean", SPLIT)).alias("sent_pos", "sentence"))
    .withColumn("sentence", F.trim("sentence"))
    .filter(F.size(F.split("sentence", " ")) >= 2))

n = sents.count()
print(f"{n} sentences -> est. {n / 17.6 / 60:.0f} min at your 17.6/sec")

25466 sentences -> est. 24 min at your 17.6/sec


## Score every sentence

Same `mapInPandas` approach as `05_inference`, but with `max_length=64` and batches of 64, since sentences are much shorter than full reviews. That made it run in 12.5 minutes, about half the estimate based on the full-review speed.

Note the model was fine-tuned on whole reviews, not single sentences, so sentence scores are less reliable than review scores, especially for very short sentences.

In [3]:
def predict_sentences(batches):
    import torch
    import pandas as pd
    from transformers import AutoTokenizer, AutoModelForSequenceClassification

    torch.set_num_threads(2)
    tok = AutoTokenizer.from_pretrained(MODEL_DIR)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).eval()

    for pdf in batches:
        texts, probs = pdf["sentence"].tolist(), []
        for i in range(0, len(texts), 64):
            enc = tok(texts[i:i + 64], truncation=True, max_length=64,
                      padding=True, return_tensors="pt")
            with torch.no_grad():
                probs.extend(torch.softmax(model(**enc).logits, dim=-1)[:, 1].tolist())
        yield pd.DataFrame({
            "review_id":  pdf["review_id"].values,
            "sent_pos":   pdf["sent_pos"].values,
            "sentence":   pdf["sentence"].values,
            "s_prob_pos": probs,
        })

SENT_SCHEMA = "review_id int, sent_pos int, sentence string, s_prob_pos double"
SENT_PATH = f"{GOLD_PATH}/sentence_predictions"

t0 = time.time()
(sents.repartition(8)
      .mapInPandas(predict_sentences, schema=SENT_SCHEMA)
      .write.mode("overwrite").parquet(SENT_PATH))
print(f"done in {(time.time() - t0) / 60:.1f} min")

done in 12.5 min


## Four-class rule

For each review, count the sentences the model is clearly sure about (≥0.8 positive or ≤0.2 positive), then apply the rules in order:

1. **mixed**: at least one clearly positive and one clearly negative sentence
2. **neutral**: the whole-review prediction has confidence under 0.6 (so this means "uncertain" more than truly neutral)
3. **positive** / **negative**: otherwise, the whole-review prediction from `05`

Result: 43% positive, 32% negative, 22% mixed, 2% neutral.

In [4]:
s = spark.read.parquet(SENT_PATH)

per_review = s.groupBy("review_id").agg(
    F.count("*").alias("n_sent"),
    F.sum((F.col("s_prob_pos") >= 0.8).cast("int")).alias("n_pos_sent"),
    F.sum((F.col("s_prob_pos") <= 0.2).cast("int")).alias("n_neg_sent"))

final = (fact.sample(fraction=SAMPLE, seed=42)

    .join(per_review, "review_id", "left")
    .fillna(0, ["n_sent", "n_pos_sent", "n_neg_sent"])
    .withColumn("sentiment_4",
        F.when((F.col("n_pos_sent") >= 1) & (F.col("n_neg_sent") >= 1), "mixed")
         .when(F.col("ar_confidence") < 0.6, "neutral")
         .when(F.col("ar_pred") == 1, "positive")
         .otherwise("negative")))

final.write.mode("overwrite").parquet(f"{GOLD_PATH}/review_sentiment_4class")
final = spark.read.parquet(f"{GOLD_PATH}/review_sentiment_4class")
final.groupBy("sentiment_4").count().orderBy(F.desc("count")).show()

+-----------+-----+
|sentiment_4|count|
+-----------+-----+
|   positive| 4412|
|   negative| 3291|
|      mixed| 2280|
|    neutral|  234|
+-----------+-----+



## Sanity check against the star rating

If the classes mean something, they should line up with the ratings:

- 5-star reviews are almost all positive (90%) and 1-star mostly negative (69%).
- **Mixed** peaks at 2 stars (32%), the rating you'd expect from a stay with real good and bad points.
- Mixed is about twice as common in negative reviews (30% of 1-2 star) as in positive ones (14% of 4-5 star): unhappy guests often still mention something they liked, while happy guests rarely add a complaint.

In [5]:
(final.groupBy("rating")
      .pivot("sentiment_4", ["positive", "negative", "mixed", "neutral"])
      .count()
      .orderBy("rating")
      .show())

+------+--------+--------+-----+-------+
|rating|positive|negative|mixed|neutral|
+------+--------+--------+-----+-------+
|     1|      65|     946|  340|     12|
|     2|     269|    2186| 1218|    110|
|     4|    1803|     134|  510|     91|
|     5|    2275|      25|  212|     21|
+------+--------+--------+-----+-------+



## Examples of mixed reviews

The sentences of five reviews labelled mixed, with each sentence's positive probability. They show the intended pattern, e.g. review 284008: `أحلى مافي هالفندق الفطور` ("the best thing in this hotel is the breakfast", 0.94) followed by `الاثاث انتهت صلاحيته` ("the furniture is past its prime", 0.06).

In [6]:
mixed_ids = final.filter("sentiment_4 = 'mixed'").select("review_id").limit(5)
(s.join(mixed_ids, "review_id")
  .orderBy("review_id", "sent_pos")
  .select("review_id", "sent_pos", F.round("s_prob_pos", 2).alias("p_pos"), "sentence")
  .toPandas())

,review_id,sent_pos,p_pos,sentence
0,88337,0,0.86,1-توقر المواقف 2- الواي فاي لا باس فيه
1,88337,1,0.18,1-الموقع بعيد عن وسط المدينة 2- الازعاج أخر الليل
2,136675,0,0.01,أسؤ أقامه
3,136675,1,0.86,الأافطار جيد الى حد ما
4,136675,2,0.00,غير نظيفأدارة فى غاية السؤ و الأاهمالالقيمة ال...
5,227016,0,0.02,أحبطت بصراحة من مستوى فندق الهيلتون
6,227016,1,0.97,السرير مريح ونظيف وقريب جدا من الحرم
7,227016,2,0.00,عدم تجاوب الإدارة عند إبلاغهم عن القصور في الخ...
8,227016,3,0.00,دون جدوىلا يستحق اسم هيلتون ولا يستحق أكثر من ...
9,284008,0,0.94,أحلى مافي هالفندق الفطور
